# PEGG 2.0 vs 3.0 — do the shared designs still agree?

Runs both versions over the same mouse variants with identical parameters and
compares the output.

Two things are being checked, and they are different questions:

1. **Backwards compatibility.** With `silent_bystander=False`, PEGG 3.0 should
   reproduce PEGG 2.0 exactly. Any difference here is a regression.
2. **The bystander designs are additive.** With `silent_bystander=True`, the
   plain designs should still match 2.0, and the extra rows should be the only
   new ones.

PEGG 2.0 is the pip-installed copy in this environment; 3.0 is the working
checkout. Both are imported in the same process under separate module names.


## Load both versions

In [1]:
import json
import os
import subprocess
import sys
import tempfile

import pandas as pd

PEGG2_ROOT = '/opt/miniconda3/envs/pegg_env/lib/python3.9/site-packages'
PEGG3_ROOT = os.path.abspath('..')

#Each version runs in its OWN process. Importing both into one interpreter does
#not work: pegg.prime does a relative "from . import bystander" at call time, so
#whichever copy is on sys.path when run() is CALLED wins, regardless of which
#module object was captured at import time.
RUNNER = r"""
import json, sys, pickle
root, in_csv, out_pkl, params_json, bystander = sys.argv[1:6]
sys.path.insert(0, root)
import pandas as pd
from pegg import prime

df = pd.read_pickle(in_csv)          # pickle, so start_end_cds stays a list
params = json.loads(params_json)

from pegg.prime import genome_loader
chrom_dict, _ = genome_loader(params.pop('_genome'))
params['chrom_dict'] = chrom_dict

if bystander == 'yes':
    params.update(silent_bystander=True, silent_per_mut=2, seed=0)

peg = prime.run(df, 'cBioPortal', **params)
peg.to_pickle(out_pkl)
print('ROWS', len(peg))
"""


def run_version(root, df, params, genome, bystander=False, label=''):
    """Run prime.run() in a subprocess under the given pegg root."""
    with tempfile.TemporaryDirectory() as tmp:
        in_csv = os.path.join(tmp, 'in.pkl')
        out_pkl = os.path.join(tmp, 'out.pkl')
        runner = os.path.join(tmp, 'runner.py')
        df.to_pickle(in_csv)
        open(runner, 'w').write(RUNNER)

        payload = dict(params)
        payload['_genome'] = genome

        proc = subprocess.run(
            [sys.executable, runner, root, in_csv, out_pkl,
             json.dumps(payload), 'yes' if bystander else 'no'],
            capture_output=True, text=True)

        if proc.returncode != 0:
            #show the child's own traceback -- otherwise the failure arrives as
            #a bare RuntimeError with the real cause discarded
            raise RuntimeError('%s failed:\n%s\n%s'
                               % (label, proc.stdout[-1500:], proc.stderr[-3000:]))
        tail = [l for l in proc.stdout.splitlines() if l.startswith('ROWS')]
        print('%-28s %s' % (label, tail[0] if tail else ''))
        return pd.read_pickle(out_pkl)


#confirm the two roots really are different versions before running anything
def _has_bystander(root):
    out = subprocess.run(
        [sys.executable, '-c',
         'import sys; sys.path.insert(0,%r); from pegg import prime; '
         'print("silent_bystander" in prime.run.__code__.co_varnames)' % root],
        capture_output=True, text=True)
    return out.stdout.strip().endswith('True')

print('PEGG 2.0 root :', PEGG2_ROOT)
print('PEGG 3.0 root :', PEGG3_ROOT)
print()
print('2.0 supports silent_bystander:', _has_bystander(PEGG2_ROOT))
print('3.0 supports silent_bystander:', _has_bystander(PEGG3_ROOT))
assert not _has_bystander(PEGG2_ROOT), 'the "2.0" copy is not 2.0'
assert _has_bystander(PEGG3_ROOT)

PEGG 2.0 root : /opt/miniconda3/envs/pegg_env/lib/python3.9/site-packages
PEGG 3.0 root : /Users/kexindong/Documents/GitHub/PEGG3.0



2.0 supports silent_bystander: False


3.0 supports silent_bystander: True


## Input

The same mouse variants used for the design library, as written by the H2M
notebook. Only the columns PEGG reads are kept, renamed from H2M's `*_m` names.


In [2]:
MUTATIONS = 'MUTATION_DESIGN.csv'

raw = pd.read_csv(MUTATIONS)
print('rows in file:', len(raw))

df = raw[raw['start_m'].notna()].copy()
df = df.rename(columns={
    'gene_name_m': 'Hugo_Symbol',
    'chr_m':       'Chromosome',
    'start_m':     'Start_Position',
    'end_m':       'End_Position',
    'ref_seq_m':   'Reference_Allele',
    'alt_seq_m':   'Tumor_Seq_Allele2',
    'type_m':      'Variant_Type',
})

#pegg keys autosomes by int and the sex chromosomes by string, matching
#genome_loader(); H2M writes 'chr3', so strip the prefix
df['Chromosome'] = df['Chromosome'].astype(str).str.replace('^chr', '', regex=True)
df['Chromosome'] = df['Chromosome'].apply(
    lambda c: c if c in ('X', 'Y') else int(c))

KEEP = ['Hugo_Symbol', 'tx_id_m', 'Chromosome', 'Start_Position', 'End_Position',
        'Reference_Allele', 'Tumor_Seq_Allele2', 'Variant_Type', 'HGVSp_m']
df = df[[c for c in KEEP if c in df.columns]].reset_index(drop=True)
df['Start_Position'] = df['Start_Position'].astype(int)
df['End_Position'] = df['End_Position'].astype(int)

print('variants to design:', len(df))
print()
print(df['Variant_Type'].value_counts().to_string())
df.head()

rows in file: 44
variants to design: 44

Variant_Type
SNP    21
INS    10
DNP     6
TNP     4
DEL     3


,Hugo_Symbol,tx_id_m,Chromosome,Start_Position,End_Position,Reference_Allele,Tumor_Seq_Allele2,Variant_Type,HGVSp_m
0,Asxl1,ENSMUST00000109790.2,2,153241365,153241366,-,G,INS,G643Wfs*14
1,Asxl1,ENSMUST00000109790.2,2,153241472,153241472,G,-,DEL,G676Efs*24
2,Asxl1,ENSMUST00000109790.2,2,153241516,153241516,C,T,SNP,R690*
3,Dnmt3a,ENSMUST00000020991.15,12,3957653,3957653,G,A,SNP,R878H
4,Dnmt3a,ENSMUST00000020991.15,12,3949640,3949640,T,G,SNP,Y532*


## Reference genome

Mouse, to match the coordinates. Loaded once and shared by both versions, so any
difference in output comes from the code rather than the input.


In [3]:
GENOME = ('/Users/kexindong/Documents/GitHub/Database/RefGenome/'
          'mouse-2023-09-13/GCF_000001635.27_GRCm39_genomic.fna.gz')

#loaded here only to validate the coordinates; each subprocess loads its own
import gzip, re
from Bio import SeqIO

chrom_dict = {}
skip = ('alternate', 'unplaced', 'unlocalized', 'patch', 'mitochondrion')
with gzip.open(GENOME, 'rt') as handle:
    for rec in SeqIO.parse(handle, 'fasta'):
        if any(w in rec.description for w in skip):
            continue
        found = re.search(r'chromosome (\w+)', rec.description)
        if not found:
            continue
        name = found.group(1)
        chrom_dict[name if name in ('X', 'Y') else int(name)] = rec.seq

print('chromosomes loaded:', len(chrom_dict))

#confirm the build before designing: a silent species/build mismatch produces
#wrong pegRNAs with no error at all
bad = []
for _, r in df.iterrows():
    if r['Variant_Type'] == 'INS':
        continue                      # REF is '-' by MAF convention
    s, e = int(r['Start_Position']), int(r['End_Position'])
    got = str(chrom_dict[r['Chromosome']][s - 1:e]).upper()
    if got != str(r['Reference_Allele']).upper():
        bad.append((r['Hugo_Symbol'], r['Chromosome'], s,
                    r['Reference_Allele'], got))

print('reference-allele mismatches:', len(bad))
for row in bad[:5]:
    print('  %s chr%s:%d expected %s, genome has %s' % row)
assert not bad, 'the genome does not match these coordinates'


chromosomes loaded: 21
reference-allele mismatches: 0


## Run both

Identical parameters. `silent_bystander` does not exist in 2.0, so it is passed
only to 3.0.


In [4]:
PARAMS = dict(
    PAM='NGG',
    RTT_lengths=[10, 15, 20, 25, 30],
    PBS_lengths=[10, 13, 15],
    pegRNAs_per_mut=10,
    min_RHA_size=1,
    RE_sites=['CGTCTC'],
    sensor=True,
    sensor_length=60,
    before_proto_context=5,
)

peg2 = run_version(PEGG2_ROOT, df, PARAMS, GENOME, label='PEGG 2.0')
peg3_off = run_version(PEGG3_ROOT, df, PARAMS, GENOME, label='PEGG 3.0 (off)')

PEGG 2.0                     ROWS 359


PEGG 3.0 (off)               ROWS 399


## 1. Backwards compatibility

With the feature off, 3.0 must reproduce 2.0. Compared on the columns that
define a design, not on the whole frame, so that added columns do not register
as differences.


In [5]:
#the columns that define a pegRNA; everything else is derived from these
DESIGN_COLS = ['Hugo_Symbol', 'Start_Position', 'PAM_start', 'PAM_strand',
               'Protospacer', 'RTT', 'PBS', 'RTT_PBS', 'Distance_to_nick',
               'RHA_size', 'PAM_disrupted', 'Proto_disrupted']
DESIGN_COLS = [c for c in DESIGN_COLS if c in peg2.columns and c in peg3_off.columns]

def key_frame(pdf):
    """Order-independent, dtype-independent view of the designs."""
    return sorted(pdf[DESIGN_COLS].astype(str).agg('|'.join, axis=1))

k2, k3 = key_frame(peg2), key_frame(peg3_off)
print('compared on:', DESIGN_COLS)
print()
print('PEGG 2.0 designs : %d' % len(k2))
print('PEGG 3.0 designs : %d' % len(k3))
print('IDENTICAL        :', k2 == k3)

if k2 != k3:
    from collections import Counter
    c2, c3 = Counter(k2), Counter(k3)
    only2, only3 = list((c2 - c3).elements()), list((c3 - c2).elements())
    print('  only in 2.0: %d' % len(only2))
    for x in only2[:3]:
        print('   ', x)
    print('  only in 3.0: %d' % len(only3))
    for x in only3[:3]:
        print('   ', x)

compared on: ['Hugo_Symbol', 'Start_Position', 'PAM_start', 'PAM_strand', 'Protospacer', 'RTT', 'PBS', 'RTT_PBS', 'Distance_to_nick', 'RHA_size', 'PAM_disrupted', 'Proto_disrupted']

PEGG 2.0 designs : 359
PEGG 3.0 designs : 399
IDENTICAL        : False
  only in 2.0: 0
  only in 3.0: 40
    Tet2|133175110|120|-|GTCTGGAGAACTGCTCCAGT|AAGACTTTCCCTTCTCAGTCCTCTA|GGAGCAGTTCTCCAG|AAGACTTTCCCTTCTCAGTCCTCTAGGAGCAGTTCTCCAG|0|22|False|True
    Tet2|133175110|120|-|GTCTGGAGAACTGCTCCAGT|AAGACTTTCCCTTCTCAGTCCTCTA|GGAGCAGTTCTCC|AAGACTTTCCCTTCTCAGTCCTCTAGGAGCAGTTCTCC|0|22|False|True
    Tet2|133175110|120|-|GTCTGGAGAACTGCTCCAGT|AAGACTTTCCCTTCTCAGTCCTTCA|GGAGCAGTTCTCCAG|AAGACTTTCCCTTCTCAGTCCTTCAGGAGCAGTTCTCCAG|0|22|False|True


### Are the scores identical too?

The sequences agreeing is the important part, but the scores should match as
well — they are computed from those sequences.


In [6]:
SCORE_COLS = [c for c in ('PEGG2_Score', 'RF_Score', 'OnTarget_Azimuth_Score')
              if c in peg2.columns and c in peg3_off.columns]

merged = peg2[DESIGN_COLS + SCORE_COLS].merge(
    peg3_off[DESIGN_COLS + SCORE_COLS], on=DESIGN_COLS,
    suffixes=('_v2', '_v3'), how='inner')
print('rows matched on design columns:', len(merged), 'of', len(peg2))

for c in SCORE_COLS:
    delta = (merged[c + '_v2'] - merged[c + '_v3']).abs()
    print('%-24s max |difference| = %.3e' % (c, delta.max()))

rows matched on design columns: 359 of 359
PEGG2_Score              max |difference| = 0.000e+00
RF_Score                 max |difference| = 0.000e+00
OnTarget_Azimuth_Score   max |difference| = 0.000e+00


## 2. With bystanders on

The plain designs should be unchanged, and the bystander rows should be purely
additional.


In [7]:
# Silent bystanders need the reading frame, so prime.run() requires the CDS
# annotation on the table -- it refuses rather than designing without it. The
# transcript is the one H2M modelled each variant onto (tx_id_m), so the frame
# matches the orthologue the mutation actually describes.
import sys
sys.path.insert(0, PEGG3_ROOT)
import gffutils
from pegg import bystander
sys.path.remove(PEGG3_ROOT)

MOUSE_DB = '/Users/kexindong/Documents/GitHub/Database/Genecode/gencode_vm33_GRCm39.db'
db_m = gffutils.FeatureDB(MOUSE_DB)

df_anno, CDS_LOOKUP = bystander.add_cds_to_variants(df, db_m, tx_column='tx_id_m')
print()
print('rows with a usable frame:', int(df_anno['cds_valid'].sum()), 'of', len(df_anno))
df_anno[['Hugo_Symbol', 'tx_id_m', 'transcript_strand', 'cds_valid']].head()

44/44 variants have a usable reading frame (19 transcripts)

rows with a usable frame: 44 of 44


,Hugo_Symbol,tx_id_m,transcript_strand,cds_valid
0,Asxl1,ENSMUST00000109790.2,+,True
1,Asxl1,ENSMUST00000109790.2,+,True
2,Asxl1,ENSMUST00000109790.2,+,True
3,Dnmt3a,ENSMUST00000020991.15,+,True
4,Dnmt3a,ENSMUST00000020991.15,+,True


In [8]:
peg3_on = run_version(PEGG3_ROOT, df_anno, PARAMS, GENOME,
                      bystander=True, label='PEGG 3.0 (on)')

plain = peg3_on[~peg3_on['has_silent_bystander']]
bys   = peg3_on[peg3_on['has_silent_bystander']]
print()
print('plain     : %d' % len(plain))
print('bystander : %d' % len(bys))

PEGG 3.0 (on)                ROWS 779

plain     : 399
bystander : 380


In [9]:
#the plain subset should still be the 2.0 design set
k_plain = sorted(plain[DESIGN_COLS].astype(str).agg('|'.join, axis=1))
print('plain designs == PEGG 2.0 :', k_plain == k2)

if k_plain != k2:
    from collections import Counter
    c2, cp = Counter(k2), Counter(k_plain)
    print('  in 2.0 but not in the plain subset: %d' % len(list((c2 - cp).elements())))
    print('  in the plain subset but not 2.0   : %d' % len(list((cp - c2).elements())))

#and no bystander row may duplicate a plain one
k_bys = set(bys[DESIGN_COLS].astype(str).agg('|'.join, axis=1))
print('bystander rows that duplicate a plain design:', len(k_bys & set(k_plain)))

plain designs == PEGG 2.0 : False
  in 2.0 but not in the plain subset: 0
  in the plain subset but not 2.0   : 40
bystander rows that duplicate a plain design: 0


## 3. Are the bystanders actually silent?

The comparison above says 3.0 did not disturb the old designs. This asks the
separate question of whether the new ones are correct: each bystander RTT is
translated in its own reading frame and must encode the same protein.


In [10]:
import Bio.Seq

# Which strand the RTT is stored on depends on PAM_strand, and the reading frame
# depends on the transcript -- neither is recoverable from the design table alone.
# Verified on two rows: for Dnmt3a the reported bystander_positions match the
# REVERSE COMPLEMENT (2;4), for Asxl1 they match the STORED string (15). So a
# blanket reverse_complement() is wrong, and this check tries both orientations
# and all three frames.
#
# That makes it a NECESSARY condition, not a sufficient one: a design that fails
# here is definitely not synonymous, but passing only means some frame works.
# test_bystander_hsc.ipynb does the authoritative check against the genome.
def _prot(seq, frame):
    s = seq[frame:]
    s = s[:len(s) // 3 * 3]
    return str(Bio.Seq.Seq(s).translate()) if s else None


def synonymous_somewhere(a, b):
    """True if a and b encode the same protein in some strand/frame combination."""
    rc = lambda s: str(Bio.Seq.Seq(s).reverse_complement())
    for x, y in ((a, b), (rc(a), rc(b))):
        for f in (0, 1, 2):
            pa, pb = _prot(x, f), _prot(y, f)
            if pa is not None and pa == pb:
                return True
    return False


groups = peg3_on.groupby(['mutation_idx', 'PAM_start', 'PAM_strand',
                          'RTT_length', 'PBS_length'])

checked = 0
not_silent = []
for _, grp in groups:
    plains = grp[~grp['has_silent_bystander']]
    others = grp[grp['has_silent_bystander']]
    if len(plains) == 0 or len(others) == 0:
        continue
    base = plains.iloc[0]['RTT']
    for _, r in others.iterrows():
        if len(r['RTT']) != len(base):
            continue                        # an indel design; lengths differ
        checked += 1
        if not synonymous_somewhere(base, r['RTT']):
            not_silent.append((r['Hugo_Symbol'], r['Start_Position'],
                               base, r['RTT'], r['bystander_positions']))

print('bystander RTTs compared to their plain counterpart:', checked)
print('not synonymous in ANY strand/frame                :', len(not_silent))
for row in not_silent[:5]:
    print('  ', row)

bystander RTTs compared to their plain counterpart: 301
not synonymous in ANY strand/frame                : 0


A frame-agnostic check is weaker than translating the whole CDS, and it can
only say that *some* frame is synonymous rather than that the transcript's own
frame is. `test_bystander_hsc.ipynb` does the authoritative version against the
genome. This one is here because it needs nothing but the design table.


## Summary

In [11]:
print('PEGG 2.0 rows              : %d' % len(peg2))
print('PEGG 3.0 rows (feature off): %d' % len(peg3_off))
print('PEGG 3.0 rows (feature on) : %d  (plain %d + bystander %d)'
      % (len(peg3_on), len(plain), len(bys)))
print()
print('1. feature off reproduces 2.0 exactly :', k2 == k3)
print('2. plain subset still matches 2.0     :', k_plain == k2)
print('3. bystanders synonymous              : %d of %d'
      % (checked - len(not_silent), checked))

PEGG 2.0 rows              : 359
PEGG 3.0 rows (feature off): 399
PEGG 3.0 rows (feature on) : 779  (plain 399 + bystander 380)

1. feature off reproduces 2.0 exactly : False
2. plain subset still matches 2.0     : False
3. bystanders synonymous              : 301 of 301
